# Streaming ERA5 ingestion and quality control

The remote corpus is never materialized. Eight-year cloud chunks are sliced to 16 anchor cells, transformed to physical units, checked, and compressed into a small reproducible cache.

In [1]:
from pathlib import Path
import json, sys
ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(ROOT / 'src'))
from pmwm.common import load_config
config = load_config(ROOT / 'config.yaml')
print(f"Project root: {ROOT}")

Project root: /home/urad/S


In [2]:
from pmwm.data import stream_era5
from pmwm.features import prepare_features
stream_era5(config)
prepare_features(config)

PosixPath('/home/urad/S/artifacts/feature_store.npz')

In [3]:
import pandas as pd
display(pd.read_csv(ROOT / 'logs/stream_blocks.csv'))

,start_year,end_year,time_steps,source_chunks_estimate,source_megabytes_estimate,retained_megabytes,elapsed_seconds,estimated_source_MB_per_second
0,1959,1966,11688,585,479.232,3.74016,12.424799,38.570605
1,1967,1974,11688,585,479.232,3.74016,12.506517,38.318583
2,1975,1982,11688,585,479.232,3.74016,12.613644,37.993144
3,1983,1990,11688,585,479.232,3.74016,12.889797,37.179175
4,1991,1998,11688,585,479.232,3.74016,12.340619,38.833708
5,1999,2006,11688,585,479.232,3.74016,12.831882,37.346977
6,2007,2014,11688,585,479.232,3.74016,12.764020,37.545538
7,2015,2022,11688,585,479.232,3.74016,12.689993,37.764561


In [4]:
manifest = json.loads((ROOT / 'artifacts/stream_manifest.json').read_text())
manifest['quality']

{'n_time': 93504,
 'n_sites': 16,
 'time_start': '1959-01-01T00:00:00.000000000',
 'time_end': '2022-12-31T18:00:00.000000000',
 'six_hour_step_fraction': 1.0,
 'monotonic_time': True,
 'duplicate_site_name_count': 0,
 'feature_checks': {'temperature_c': {'finite_fraction': 1.0,
   'minimum': -35.98291015625,
   'maximum': 40.808746337890625,
   'outside_plausible_fraction': 0.0},
  'pressure_hpa': {'finite_fraction': 1.0,
   'minimum': 936.0262451171875,
   'maximum': 1060.027099609375,
   'outside_plausible_fraction': 0.0},
  'u_wind_ms': {'finite_fraction': 1.0,
   'minimum': -17.801860809326172,
   'maximum': 18.729413986206055,
   'outside_plausible_fraction': 0.0},
  'v_wind_ms': {'finite_fraction': 1.0,
   'minimum': -17.976694107055664,
   'maximum': 17.82708740234375,
   'outside_plausible_fraction': 0.0},
  'precipitation_mm': {'finite_fraction': 1.0,
   'minimum': 0.0,
   'maximum': 40.31277847290039,
   'outside_plausible_fraction': 0.0}}}